# Od napovedi do poslovne odločitve — B: vaja (telekom)
**Akademija UI za poslovne aplikacije — Modul 2, Uvod v strojno učenje**

Pokličemo lahko 100 strank. Model drevesa jih izbere tako, da jih 86 res odide.
**Ali ga lahko premagate s svojim pravilom?**

### Ponovitev
- Model vsaki stranki pripiše **verjetnost odhoda** (`predict_proba`).
- Pri omejenem proračunu (100 klicev) strank ne delimo s pragom 0,5. **Razvrstimo** jih po verjetnosti in vzamemo vrh seznama.
- Isto vprašanje zastavimo vsakemu modelu: **koliko od izbranih 100 strank je res odšlo?** Tako dobimo lestvico.
- **Tveganje ni isto kot korist:** visoka verjetnost odhoda ne pove, ali bi naša ponudba stranko zadržala.
- **Mere iz prejšnjega sklopa** (točnost, natančnost, priklic) ocenijo vse napovedi pri pragu 0,5. Pri 100 klicih šteje samo vrh seznama.

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
shema = preberi_shemo(PODATKI / "shema.json")


### Knjižnice, podatki in modela

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from sklearn.tree import DecisionTreeClassifier, export_text

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME, TELEKOM_STEVILSKE
from utils.orodja import izberi_najbolj_tvegane, vrstica_lestvice, graf_izbranih
nastavi_slog()

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je pripravila skripta `scripts/pripravi_telekom.py`. Naredi isto kot zvezka predavatelja
`A1_Priprava_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/01_priprava_podatkov`) in
`A1_Delitev_in_manjkajoce_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/02_preverjanje_modela`).
Tam so vse odločitve z razlogi. Na kratko:

1. Ena vrstica = ena stranka (100 000 strank). Cilj je `odhod`. `id_stranke` ni značilka.
2. Tabela ima 64 značilk: 58 številskih in 6 besedilnih. Stolpcem iz 1. modula so dodani novi
   (3- in 6-mesečna povprečja, razmerja, logaritmi).
3. Skrite oznake (`U`, `UNKW`) in nemogoče negativne vrednosti so postale manjkajoče vrednosti.
4. Delitev 60 / 20 / 20: naključna, z enakim deležem odhodov, seme 42. Vsi imamo iste stranke v istih množicah.
5. Manjkajoče številke so zapolnjene z **mediano učne množice**. Stolpci `..._manjka` povedo, kje je vrednost manjkala.
6. Besedilni stolpci so zapisani tudi kot stolpci 0/1 (na primer `regija_CHICAGO_AREA`).
   Skupaj je 96 stolpcev, ki jih lahko podamo modelu.
7. Osamelcev ne odstranimo in značilk ne skaliramo: drevo vpraša samo, ali je vrednost nad ali pod pragom.
8. Uporabimo 16 številskih značilk iz 1. modula, kot pri drevesu. Z istimi značilkami bodo na lestvico prišli tudi naslednji modeli.
9. Modele primerjamo na validacijski množici. Testna ostane zaprta do tekmovanja na koncu klasifikacijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)

ZNACILKE = TELEKOM_STEVILSKE                                           # 16 številskih značilk iz 1. modula
X_ucni, y_ucni = ucni[ZNACILKE], ucni["odhod"]
X_val, y_val = validacijski[ZNACILKE], validacijski["odhod"]
print(f"Učnih strank: {len(ucni)}, validacijskih: {len(validacijski)}, od teh je odšlo {y_val.mean():.1%}")
print(f"Stolpcev v tabeli: {ucni.shape[1]}, manjkajočih vrednosti: {int(ucni.isna().sum().sum())}")

### Model iz prejšnjega sklopa
Drevo globine 5 naučimo na istih 16 številskih značilkah kot v sklopu o drevesu. Skaliranja ne potrebuje:
vrednost samo primerja s pragom.

In [ ]:
drevo = DecisionTreeClassifier(max_depth=5, random_state=SEME).fit(X_ucni, y_ucni)

# verjetnost odhoda za vsako validacijsko stranko
verjetnosti_drevo = pd.Series(drevo.predict_proba(X_val)[:, 1], index=X_val.index)
print(f"Drevo je naučeno. Validacijska točnost: {drevo.score(X_val, y_val):.3f}")

---
## B1. Vaše pravilo proti modelu
Izberite en stolpec in smer. Celica izbere 100 strank z največjo (ali najmanjšo) vrednostjo in pokaže, koliko jih je odšlo.

> **Povezava s 1. modulom:** pravilo poiščite v tem, kar smo videli na grafih. Pri katerih vrednostih je bil delež
> odhodov največji?

Poskusite: `"dni_z_napravo"` (največ), `"mesecev_narocnine"` (najmanj), `"povp_minute"` (najmanj),
`"cena_naprave"` (najmanj), `"sprememba_minut"` (najmanj).

Uporabite lahko tudi nove številske stolpce, na primer `"razmerje_minut_trenutno_6m"` (minute zdaj, deljene s
6-mesečnim povprečjem) ali `"povp_minute_3m"`. Vsa imena izpiše `list(validacijski.columns)`.

**Vprašanja:**
1. Katero pravilo pride najbližje drevesu (86)? Ali ga katero premaga?
2. Katero pravilo je slabše od naključnega izbora (50)? Zakaj?

In [ ]:
STOLPEC = "dni_z_napravo"     # ← zamenjajte
NAJVECJE = True               # ← True: 100 strank z NAJVEČJO vrednostjo; False: z NAJMANJŠO

ocene = validacijski[STOLPEC] if NAJVECJE else -validacijski[STOLPEC]
izbrane = izberi_najbolj_tvegane(ocene, 100)

print(f"Vaše pravilo: {int(y_val.loc[izbrane].sum())} od 100.   "
      f"Model drevesa: {vrstica_lestvice('', verjetnosti_drevo, y_val)['odšlo od 100']} od 100.   "
      f"Naključno: približno {100 * y_val.mean():.0f} od 100.")
graf_izbranih({f"{'Največ' if NAJVECJE else 'Najmanj'}: {STOLPEC}": y_val.loc[izbrane],
               "Model: drevo (globina 5)": y_val.loc[izberi_najbolj_tvegane(verjetnosti_drevo, 100)]})

<details>
<summary><b>Rešitev (klikni)</b></summary>

1. Vsi spodnji rezultati so na **isti validacijski množici**: koliko od 100 izbranih strank je res odšlo.
   Pravilo **najmanj minut** izbere 84 takih strank, pravilo **najnižja cena naprave** 63.
   Nobeno od petih preizkušenih pravil ne preseže drevesa (86). Izenači ga pravilo na novem stolpcu:
   **najmanjše** `razmerje_minut_trenutno_6m` (86 od 100). To so stranke, ki telefon zdaj uporabljajo veliko manj
   kot v zadnjih šestih mesecih. Podobno pravilo je drevo našlo samo (`povp_minute <= 0,38`).
2. Pravilo **najmanj mesecev naročnine** izbere le 31 od 100 strank, ki odidejo; naključno bi jih v povprečju izbrali približno 50.
   Nove stranke odhajajo redkeje; skok pride šele pri 11 mesecih, kar smo videli v 1. modulu.

Pri 1 000 izbranih strankah je vrstni red drugačen: pravilo "najmanj minut" najde 632 odhodov,
"najmanjše razmerje minut" 686, drevo pa 720. Strank skoraj brez minut je premalo; ko jih zmanjka, pravilo ne ve več, koga izbrati.
Model uporablja vse značilke hkrati.

</details>

Preverimo: predlagana pravila, pravilo na novem stolpcu in oba modela, za 100 in za 1 000 strank.

In [ ]:
print(export_text(drevo, feature_names=ZNACILKE, max_depth=1))          # prvi dve ravni vprašanj drevesa

pravila = [("dni_z_napravo", True), ("mesecev_narocnine", False), ("povp_minute", False),
           ("cena_naprave", False), ("sprememba_minut", False), ("razmerje_minut_trenutno_6m", False)]

vrstice = [{"izbira": "naključno (pričakovano)", "odšlo od 100": round(100 * y_val.mean()), "odšlo od 1 000": round(1000 * y_val.mean())}]
for stolpec, najvecje in pravila:
    ocene = validacijski[stolpec] if najvecje else -validacijski[stolpec]
    vrstice.append(vrstica_lestvice(f"ročno: {'največ' if najvecje else 'najmanj'} {stolpec}", ocene, y_val))
vrstice.append(vrstica_lestvice("drevo (globina 5)", verjetnosti_drevo, y_val))

pd.DataFrame(vrstice).sort_values("odšlo od 1 000")

**Za razpravo:** pravila primerjamo na isti validacijski množici. Če med mnogimi pravili izberemo najboljše po njej,
rezultat na tej množici ni več neodvisna končna ocena; testna množica ostane zaprta.
Model označi stranke z večjim tveganjem odhoda v tem izvlečku. Ali to pomeni, da moramo poklicati prav te stranke?
Katere podatke bi potrebovali, da bi vedeli, koga lahko s ponudbo zadržimo?

---
### Za hitre: zakaj globlje drevo izbere slabši seznam?
V zvezku A je imelo drevo globine 8 večjo točnost kot drevo globine 5, med prvimi 100 strankami pa jih je našlo manj (72 proti 86).
Poskusite `GLOBINA = 8` in `GLOBINA = 20`. Poglejte, koliko strank dobi verjetnost 1,00.

In [ ]:
GLOBINA = 5          # ← zamenjajte: 8, 20

d = DecisionTreeClassifier(max_depth=GLOBINA, random_state=SEME).fit(X_ucni, y_ucni)
verjetnosti = pd.Series(d.predict_proba(X_val)[:, 1], index=X_val.index)
print(f"Globina {GLOBINA}: točnost {d.score(X_val, y_val):.3f},", vrstica_lestvice("", verjetnosti, y_val))
print("Strank z verjetnostjo 1,00:", int((verjetnosti == 1).sum()))

<details>
<summary><b>Rešitev (klikni)</b></summary>

Obe drevesi naučimo na 60 000 učnih strankah in ju ocenimo na istih 20 000 **validacijskih** strankah.
**Točnost** meri delež pravilnih napovedi »odide« ali »ostane« med vsemi 20 000: pri globini 8 je 0,606, pri globini 5 pa 0,596.

Lestvica meri nekaj drugega: stranke razvrstimo po napovedani verjetnosti odhoda in pogledamo samo **prvih 100**.
Pri globini 8 jih je res odšlo 72, pri globini 5 pa 86. Boljša skupna točnost torej še ne pomeni boljšega seznama za 100 klicev.

Globoko drevo ustvari majhne liste. Če so v učnem listu odšle vse stranke, tudi validacijskim strankam v tem listu
pripiše verjetnost **1,00**. Pri globini 8 je takih 66 validacijskih strank, vendar jih je res odšlo le 48.
Pri globini 20 enako oceno 1,00 dobi kar 5 823 strank; od teh jih je res odšlo približno 55 %.
Ker imajo enako oceno, program med njimi izbere prvih 100 po dogovorjenem naključnem vrstnem redu (seme 42);
v tem izboru jih odide 50. To kaže, da so verjetnosti globokega drevesa preveč samozavestne.
Večji `min_samples_leaf` omeji zelo majhne liste in lahko zmanjša takšno pretirano prilagajanje.

</details>

Preverimo: točnost, lestvica in število strank z verjetnostjo 1,00 za globine 3, 5, 8 in 20.

In [ ]:
for globina in [3, 5, 8, 20]:
    d = DecisionTreeClassifier(max_depth=globina, random_state=SEME).fit(X_ucni, y_ucni)
    v = pd.Series(d.predict_proba(X_val)[:, 1], index=X_val.index)
    print(f"globina {globina:2d}: točnost {d.score(X_val, y_val):.3f}", vrstica_lestvice("", v, y_val),
          "strank z verjetnostjo 1,00:", int((v == 1).sum()))